In [1]:
# !pip install optuna lightgbm pandas scikit-learn matplotlib

In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

# Load data (replace with your dataset path)
X_train = pd.read_parquet('temp/X_resampled.parquet')
X_val = pd.read_parquet('temp/X_val.parquet')


y_train = X_train['TARGET']
y_val = X_val['TARGET']

X_train = X_train.drop(columns=['TARGET'])
X_val = X_val.drop(columns=['TARGET'])

In [3]:
X_train.columns = X_train.columns.str.replace(r'[^\w]', '_', regex=True)
X_val.columns = X_val.columns.str.replace(r'[^\w]', '_', regex=True)

In [4]:
import optuna
import lightgbm as lgb
import numpy as np

def objective(trial):
    param = {
        'objective': 'binary',
        'metric': 'auc',
        'boosting_type': 'gbdt',
        'learning_rate': trial.suggest_float('learning_rate', 1e-3, 2e-2, log=True),
        'num_leaves': trial.suggest_int('num_leaves', 50, 200),
        'max_depth': trial.suggest_int('max_depth', 10, 15),
        'min_data_in_leaf': trial.suggest_int('min_data_in_leaf', 100, 200),
        'feature_fraction': trial.suggest_float('feature_fraction', 0.4, 0.8),
        'bagging_fraction': trial.suggest_float('bagging_fraction', 0.7, 1.0),
        'bagging_freq': trial.suggest_int('bagging_freq', 1, 7),
        'lambda_l1': trial.suggest_float('lambda_l1', 1e-3, 10.0, log=True),
        'lambda_l2': trial.suggest_float('lambda_l2', 1e-3, 10.0, log=True),
        "min_child_samples": trial.suggest_int("min_child_samples", 50, 200),
        'n_jobs': -1
    }
    
    train_data = lgb.Dataset(X_train, label=y_train)
    valid_data = lgb.Dataset(X_val, label=y_val, reference=train_data)
    
    model = lgb.train(
        param,
        train_data,
        valid_sets=[valid_data],
        num_boost_round=1000,
    )
    
    preds = model.predict(X_val)
    auc = roc_auc_score(y_val, preds)
    return auc


In [5]:
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=100)

# Print the best parameters
print("Best parameters:", study.best_params)
print("Best AUC:", study.best_value)

[I 2024-12-03 01:09:13,169] A new study created in memory with name: no-name-a98a2210-c292-4b36-bf5b-fdf3e6f12a8b
/tmp/ipykernel_850018/624360741.py:10: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  'learning_rate': trial.suggest_loguniform('learning_rate', 1e-3, 2e-2),
/tmp/ipykernel_850018/624360741.py:14: FutureWarning: suggest_uniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float instead.
  'feature_fraction': trial.suggest_uniform('feature_fraction', 0.4, 0.8),
/tmp/ipykernel_850018/624360741.py:15: FutureWarning: suggest_uniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float instead.
  'bagging_fraction': trial.suggest_unifo

[LightGBM] [Warning] min_data_in_leaf is set=167, min_child_samples=61 will be ignored. Current value: min_data_in_leaf=167
[LightGBM] [Warning] min_data_in_leaf is set=167, min_child_samples=61 will be ignored. Current value: min_data_in_leaf=167
[LightGBM] [Info] Number of positive: 246009, number of negative: 226133
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.759568 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 154817
[LightGBM] [Info] Number of data points in the train set: 472142, number of used features: 975
[LightGBM] [Warning] min_data_in_leaf is set=167, min_child_samples=61 will be ignored. Current value: min_data_in_leaf=167
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.521049 -> initscore=0.084245
[LightGBM] [Info] Start training from score 0.084245


[I 2024-12-03 01:12:47,204] Trial 0 finished with value: 0.7854240567994382 and parameters: {'learning_rate': 0.007550976757859301, 'num_leaves': 50, 'max_depth': 14, 'min_data_in_leaf': 167, 'feature_fraction': 0.7068591743586025, 'bagging_fraction': 0.9117096069320529, 'bagging_freq': 1, 'lambda_l1': 3.268621639965369, 'lambda_l2': 0.0010299589511021318, 'min_child_samples': 61}. Best is trial 0 with value: 0.7854240567994382.
/tmp/ipykernel_850018/624360741.py:10: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  'learning_rate': trial.suggest_loguniform('learning_rate', 1e-3, 2e-2),
/tmp/ipykernel_850018/624360741.py:14: FutureWarning: suggest_uniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float instead.
  'feature_fraction': t

[LightGBM] [Warning] min_data_in_leaf is set=115, min_child_samples=156 will be ignored. Current value: min_data_in_leaf=115
[LightGBM] [Warning] min_data_in_leaf is set=115, min_child_samples=156 will be ignored. Current value: min_data_in_leaf=115
[LightGBM] [Info] Number of positive: 246009, number of negative: 226133
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.906959 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 154819
[LightGBM] [Info] Number of data points in the train set: 472142, number of used features: 976
[LightGBM] [Warning] min_data_in_leaf is set=115, min_child_samples=156 will be ignored. Current value: min_data_in_leaf=115
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.521049 -> initscore=0.084245
[LightGBM] [Info] Start training from score 0.084245
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive

[I 2024-12-03 01:15:17,608] Trial 1 finished with value: 0.7882614254524937 and parameters: {'learning_rate': 0.015671033802343113, 'num_leaves': 74, 'max_depth': 11, 'min_data_in_leaf': 115, 'feature_fraction': 0.5047812309163111, 'bagging_fraction': 0.8122537927681251, 'bagging_freq': 1, 'lambda_l1': 0.2208759992127922, 'lambda_l2': 0.0026182719311654695, 'min_child_samples': 156}. Best is trial 1 with value: 0.7882614254524937.
/tmp/ipykernel_850018/624360741.py:10: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  'learning_rate': trial.suggest_loguniform('learning_rate', 1e-3, 2e-2),
/tmp/ipykernel_850018/624360741.py:14: FutureWarning: suggest_uniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float instead.
  'feature_fraction':

[LightGBM] [Warning] min_data_in_leaf is set=166, min_child_samples=57 will be ignored. Current value: min_data_in_leaf=166
[LightGBM] [Warning] min_data_in_leaf is set=166, min_child_samples=57 will be ignored. Current value: min_data_in_leaf=166
[LightGBM] [Info] Number of positive: 246009, number of negative: 226133
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.835014 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 154817
[LightGBM] [Info] Number of data points in the train set: 472142, number of used features: 975
[LightGBM] [Warning] min_data_in_leaf is set=166, min_child_samples=57 will be ignored. Current value: min_data_in_leaf=166
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.521049 -> initscore=0.084245
[LightGBM] [Info] Start training from score 0.084245


[I 2024-12-03 01:20:13,856] Trial 2 finished with value: 0.7821703430407195 and parameters: {'learning_rate': 0.004293906832707552, 'num_leaves': 168, 'max_depth': 14, 'min_data_in_leaf': 166, 'feature_fraction': 0.5485003638828079, 'bagging_fraction': 0.7770519999425093, 'bagging_freq': 4, 'lambda_l1': 0.08643500904033433, 'lambda_l2': 2.6566795920202777, 'min_child_samples': 57}. Best is trial 1 with value: 0.7882614254524937.
/tmp/ipykernel_850018/624360741.py:10: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  'learning_rate': trial.suggest_loguniform('learning_rate', 1e-3, 2e-2),
/tmp/ipykernel_850018/624360741.py:14: FutureWarning: suggest_uniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float instead.
  'feature_fraction': t

[LightGBM] [Warning] min_data_in_leaf is set=114, min_child_samples=90 will be ignored. Current value: min_data_in_leaf=114
[LightGBM] [Warning] min_data_in_leaf is set=114, min_child_samples=90 will be ignored. Current value: min_data_in_leaf=114
[LightGBM] [Info] Number of positive: 246009, number of negative: 226133
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.873207 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 154819
[LightGBM] [Info] Number of data points in the train set: 472142, number of used features: 976
[LightGBM] [Warning] min_data_in_leaf is set=114, min_child_samples=90 will be ignored. Current value: min_data_in_leaf=114
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.521049 -> initscore=0.084245
[LightGBM] [Info] Start training from score 0.084245


In [ ]:
{'learning_rate': 0.009407721399062104, 'num_leaves': 198, 'max_depth': 13, 'min_data_in_leaf': 99, 'feature_fraction': 0.5038532461662355, 'bagging_fraction': 0.9912874057611964, 'bagging_freq': 4, 'lambda_l1': 8.040904175809203, 'lambda_l2': 0.1439055371441105, 'min_child_samples': 196}
{'learning_rate': 0.010857267239676163, 'num_leaves': 148, 'max_depth': 15, 'min_data_in_leaf': 161, 'feature_fraction': 0.553966238029556, 'bagging_fraction': 0.8089748461247909, 'bagging_freq': 5, 'lambda_l1': 7.742831278633271, 'lambda_l2': 3.5102491435855927, 'min_child_samples': 177}
{'learning_rate': 0.013890451501153814, 'num_leaves': 147, 'max_depth': 15, 'min_data_in_leaf': 163, 'feature_fraction': 0.5670479460293333, 'bagging_fraction': 0.809182514501803, 'bagging_freq': 5, 'lambda_l1': 7.925377670194449, 'lambda_l2': 3.596720591358153, 'min_child_samples': 178}
{'learning_rate': 0.011504291774263705, 'num_leaves': 178, 'max_depth': 15, 'min_data_in_leaf': 159, 'feature_fraction': 0.5324001102890034, 'bagging_fraction': 0.8146135861371512, 'bagging_freq': 5, 'lambda_l1': 7.581976219371397, 'lambda_l2': 7.190965593534284, 'min_child_samples': 169}

{'learning_rate': 0.011210635312865506, 'num_leaves': 163, 'max_depth': 15, 'min_data_in_leaf': 172, 'feature_fraction': 0.5716044278384449, 'bagging_fraction': 0.7884948548078659, 'bagging_freq': 5, 'lambda_l1': 5.991689370702647, 'lambda_l2': 2.8907075909353415, 'min_child_samples': 39}

{'learning_rate': 0.012904378053854955, 'num_leaves': 111, 'max_depth': 15, 'min_data_in_leaf': 168, 'feature_fraction': 0.5393696234146808, 'bagging_fraction': 0.778028657451273, 'bagging_freq': 5, 'lambda_l1': 9.936978036207751, 'lambda_l2': 3.1813863542002587, 'min_child_samples': 100} # 152 0.5966
{'learning_rate': 0.016557701261941987, 'num_leaves': 75, 'max_depth': 15, 'min_data_in_leaf': 189, 'feature_fraction': 0.6203276221518776, 'bagging_fraction': 0.8406049960487956, 'bagging_freq': 5, 'lambda_l1': 9.82248801299947, 'lambda_l2': 1.0969275796157616, 'min_child_samples': 96} #166 0.598


In [25]:
best_params = study.best_params
best_params['objective'] = 'binary'
best_params['metric'] = 'auc'

In [ ]:
best_params

In [ ]:
best_params = best_params
best_params['objective'] = 'binary'
best_params['metric'] = 'auc'

print("Training the final model with the best parameters")
print(best_params)

final_model = lgb.train(
    best_params,
    lgb.Dataset(X_train, label=y_train),
    num_boost_round=1000
)

In [27]:
y_pred = final_model.predict(X_val)

In [ ]:
roc_auc_score(y_val,y_pred)

In [ ]:
# Get feature importance and feature names
importance = final_model.feature_importance(importance_type='gain')  # 'gain' measures the contribution
importance_split = final_model.feature_importance(importance_type='split')
feature_names = X_train.columns

# Create a DataFrame for better visualization
importance_df = pd.DataFrame({
    'Feature': feature_names,
    'Importance': importance,
    'Num_Split': importance_split
}).sort_values(by='Importance', ascending=False)

# Display the most important feature
best_feature = importance_df.iloc[0]
print(f"The most important feature is: {best_feature['Feature']} with an importance score of {best_feature['Importance']}")

# Optional: Display the top 5 features
print("\nTop 5 Features:")
print(importance_df.head())


In [36]:
importance_df.to_excel('temp/feature_importance.xlsx', index=False)